# Model Evaluation

## Set Up

In [1]:
import sys, os

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, SEED
from src.data_utils import export_data

from pathlib import Path
import pandas as pd

## Run eval

In [ ]:
def make_cmd(
    outcome,
    model_name,
    model_imp_dir,
    data_imp_dir,
    results_dir,
    n_bootstraps,
    n_bins,
    show_progress,
    metric_list,
    seed,
):
    metrics_strs = " ".join(metric_list)
    cmd_str = f"export PYTHONPATH={BASE_PATH}; \
                export OMP_NUM_THREADS={4}; \
                uv run python -m src.eval \
                --outcome {outcome} \
                --model_name {model_name} \
                --model_imp_dir {model_imp_dir} \
                --data_imp_dir {data_imp_dir} \
                --results_dir {results_dir} \
                --n_bootstraps {n_bootstraps} \
                --n_bins {n_bins} \
                --show_progress {show_progress} \
                --metrics_strs {metrics_strs} \
                --seed {seed}"
    return " ".join(cmd_str.split())

In [ ]:
SWARM_DIR = BASE_PATH / "swarm/eval"
outcome_list = [
    "SERIOUS",
    "ANY",
    "PNEUMO",
    "CARDIAC_COMP",
    "VTE",
    "SEPSIS",
    "SSI",
    "UTI",
    "RENAL",
    "UNPLNREOP",
    "MORT",
]
model_list = ["lgbm", "lr", "xgb", "nn", "stack"]

In [ ]:
full_cmd_list = []
for model in model_list:
    swarm_path = SWARM_DIR / f"cmds/{model}.swarm"
    if swarm_path.exists():
        swarm_path.unlink()
    swarm_path.parent.mkdir(exist_ok=True, parents=True)
    model_cmd_list = []
    for outcome in outcome_list:
        cmd = make_cmd(
            outcome=outcome,
            model_name=model,
            model_imp_dir=BASE_PATH / "models/calibrated",
            data_imp_dir=BASE_PATH / "data/processed",
            results_dir=BASE_PATH / "results",
            n_bootstraps=5000,
            n_bins=4,
            show_progress=False,
            metric_list=["f1", "accuracy", "recall", "precision", "brier", "ici"],
            seed=SEED,
        )
        model_cmd_list.append(cmd)
    swarm_path.write_text("\n".join(model_cmd_list))
    full_cmd_list += model_cmd_list
print(f"Number of cmds: {len(full_cmd_list)}")

In [ ]:
# all run in similar times, so don't need model-specific config
config_dict = {
    model_name: {
        "swarm_time": "1:00:00",
        "gb": 2,
        "partition": "quick",
    }
    for model_name in model_list
}

Run sequentially

In [ ]:
# import subprocess

# for iteration, cmd in enumerate(full_cmd_list):
#     print(f"{iteration +1}/{len(full_cmd_list)}...")
#     subprocess.run(cmd, shell=True, check=True)

Run in parallel

In [ ]:
from src.swarm import run_tune_eval_swarm

run_tune_eval_swarm(
    model_list=model_list,
    log_base_dir=SWARM_DIR / "logs",
    cmd_dir=SWARM_DIR / "cmds",
    n_threads=1,
    config_dict=config_dict,
)

## Aggregate table results

In [2]:
imp_dir = BASE_PATH / "results/tables"
bin_dir = imp_dir / "bins"
metric_dir = imp_dir / "metrics"

Bins

In [3]:
for outcome_dir in bin_dir.iterdir():
    df_list = []
    export_path = bin_dir / "combined" / f"{outcome_dir.name}.xlsx"
    for model_path in outcome_dir.iterdir():
        df = pd.read_excel(model_path, index_col=0)
        df_list.append(df)
    df_combined = pd.concat(df_list)
    export_data(data_to_export=df_combined, export_path=Path(export_path))

Metrics

In [ ]:
dev_list = []
test_list = []
## Aggregate dev and test into large comprehensive dfs
for outcome_dir in metric_dir.iterdir():
    for model_path in outcome_dir.iterdir():
        df = pd.read_excel(model_path, index_col=0)
        df.insert(0, "outcome", outcome_dir.name)
        df_dev = df[df["Cohort"] != "test"]
        dev_list.append(df_dev)
        df_test = df[df["Cohort"] == "test"]
        test_list.append(df_test)

dev_combined = pd.concat(dev_list)
dev_combined = dev_combined.sort_values(by=["outcome", "Cohort", "Model"]).reset_index(
    drop=True
)
export_data(data_to_export=dev_combined, export_path=metric_dir / "dev_combined.xlsx")

test_combined = pd.concat(test_list)
test_combined = test_combined.sort_values(by=["outcome", "Model"]).reset_index(
    drop=True
)
export_data(data_to_export=test_combined, export_path=metric_dir / "test_combined.xlsx")